In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.float_format", "{:.3f}".format)

df = pd.read_csv("RIPAoc2020.csv", low_memory=False)

# basic cleaning
df["DATE_OF_STOP"] = pd.to_datetime(df["DATE_OF_STOP"], errors="coerce")
df = df[df["DATE_OF_STOP"].dt.year == 2020].copy()

# ensure stop reason is string
df["REASON_FOR_STOP"] = df["REASON_FOR_STOP"].astype(str).str.upper()

/var/folders/4n/gr221w3n74g83smf7w9pzfy00000gn/T/ipykernel_12525/108664364.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["DATE_OF_STOP"] = pd.to_datetime(df["DATE_OF_STOP"], errors="coerce")


In [2]:
race_cols = {
    "White": "RAE_WHITE",
    "Black": "RAE_BLACK_AFRICAN_AMERICAN",
    "Hispanic": "RAE_HISPANIC_LATINO",
    "Asian": "RAE_ASIAN",
    "Other/Unknown": "RAE_MULTIRACIAL"
}

# ensure numeric
for c in race_cols.values():
    df[c] = pd.to_numeric(df[c], errors="coerce")

def get_race(row):
    for race, col in race_cols.items():
        if row[col] == 1:
            return race
    return np.nan

df["RACE"] = df.apply(get_race, axis=1)
df = df[df["RACE"].notna()].copy()

In [4]:
rs_cols = [
    "RFS_RS_REASON_SUSP",
    "RFS_RS_VIOLENT_CRIME",
    "RFS_RS_DRUG_TRANS",
    "RFS_RS_MATCH_SUSPECT"
]

for c in rs_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

df["MOVING"] = df["REASON_FOR_STOP"].str.contains("MOVING", na=False)
df["EQUIPMENT"] = df["REASON_FOR_STOP"].str.contains("EQUIP", na=False)

df["INVESTIGATIVE"] = (
    (df["RFS_RS_REASON_SUSP"] == 1) |
    (df["RFS_RS_VIOLENT_CRIME"] == 1) |
    (df["RFS_RS_DRUG_TRANS"] == 1) |
    (df["RFS_RS_MATCH_SUSPECT"] == 1)
)

In [5]:
table1 = df.groupby("RACE").agg(
    total_stops=("RACE", "size"),
    moving=("MOVING", "mean"),
    equipment=("EQUIPMENT", "mean"),
    investigative=("INVESTIGATIVE", "mean")
)

# as percent
table1[["moving", "equipment", "investigative"]] *= 100

# percent of all stops
table1["pct_stops"] = table1["total_stops"] / len(df) * 100

# reorder
table1 = table1[[
    "total_stops",
    "pct_stops",
    "moving",
    "equipment",
    "investigative"
]]

table1.columns = [
    "Total Stops",
    "% of Stops",
    "Moving (%)",
    "Equipment (%)",
    "Investigative (%)"
]

table1

,Total Stops,% of Stops,Moving (%),Equipment (%),Investigative (%)
RACE,,,,,
Asian,2520,6.785,0.000,0.000,2.421
Black,1585,4.268,0.000,0.000,8.833
Hispanic,13612,36.651,0.000,0.000,4.834
White,19423,52.297,0.000,0.000,5.241


In [6]:
total_row = pd.DataFrame({
    "Total Stops": [len(df)],
    "% of Stops": [100.0],
    "Moving (%)": [df["MOVING"].mean() * 100],
    "Equipment (%)": [df["EQUIPMENT"].mean() * 100],
    "Investigative (%)": [df["INVESTIGATIVE"].mean() * 100],
}, index=["Total"])

table1_final = pd.concat([table1, total_row])

table1_final

,Total Stops,% of Stops,Moving (%),Equipment (%),Investigative (%)
Asian,2520,6.785,0.000,0.000,2.421
Black,1585,4.268,0.000,0.000,8.833
Hispanic,13612,36.651,0.000,0.000,4.834
White,19423,52.297,0.000,0.000,5.241
Total,37140,100.000,0.000,0.000,5.054


In [7]:
print(
    table1_final.to_latex(
        float_format="%.1f",
        caption="Stop Characteristics by Race, Orange County Sheriff's Department (2020)",
        label="tab:summary"
    )
)

\begin{table}
\caption{Stop Characteristics by Race, Orange County Sheriff's Department (2020)}
\label{tab:summary}
\begin{tabular}{lrrrrr}
\toprule
 & Total Stops & % of Stops & Moving (%) & Equipment (%) & Investigative (%) \\
\midrule
Asian & 2520 & 6.8 & 0.0 & 0.0 & 2.4 \\
Black & 1585 & 4.3 & 0.0 & 0.0 & 8.8 \\
Hispanic & 13612 & 36.7 & 0.0 & 0.0 & 4.8 \\
White & 19423 & 52.3 & 0.0 & 0.0 & 5.2 \\
Total & 37140 & 100.0 & 0.0 & 0.0 & 5.1 \\
\bottomrule
\end{tabular}
\end{table}

